In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
sys.path.append("..")
sys.path.append("../..")

from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel, FinalAnswerTool, GoogleSearchTool, VisitWebpageTool
import yaml
import PIL
import numpy as np
import torchvision.transforms as T

import torch

from agents.utils import export_masks

from src.kitti_tracking_hf import KittiHFIterableDataset

from matplotlib import pyplot as plt

/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
root_dir = "/mnt/ssd/kitti_tracking"
n_steps, n_pred_steps = 8, 0

dataset_builder = KittiHFIterableDataset(
	root_dir=root_dir,
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
		T.Resize((120, 320)),
	])
)
dataset = dataset_builder.to_hf_dataset()
for i, sample in enumerate(dataset):
    break

In [4]:
web_search = GoogleSearchTool()
# search_tool = DuckDuckGoSearchTool(
#     max_results=5,
# )
visit_webpage = VisitWebpageTool()

In [ ]:
pages = web_search(
    "What is Indoor Service Robot? What sensing modalities are necessary? What object classes are of interest?"
)
print(pages)

In [ ]:
urls = [
	"https://standardbots.com/blog/every-type-of-sensors-in-robotics---explained",
	"https://pmc.ncbi.nlm.nih.gov/articles/PMC10893033/",
	"https://www.sciencedirect.com/topics/computer-science/service-robot",
	"https://acroname.com/blog/sensors-robotics-5-common-types-0?srsltid=AfmBOor_I7R_vvfEdjwpbSzVXVJOFQcp5h5SgxinJwV9Wt-v0yA4a1nn"
	"https://ifr.org/img/office/Service_Robots_2016_Chapter_1_2.pdf"
]
for url in urls:
	print(url)
	whole_page = visit_webpage(url)
	
	print(whole_page)
	break

In [14]:
model = HfApiModel(
	max_tokens=4906,
	temperature=0.5,
	model_id="meta-llama/Meta-Llama-3-8B-Instruct", # it is possible that this model may be overloaded
	custom_role_conversions=None,
)

with open("./interpreter.yaml", 'r') as stream:
	prompt_templates = yaml.safe_load(stream)
# search_tool = DuckDuckGoSearchTool()
final_answer = FinalAnswerTool()

# final_answer 
agent = CodeAgent(
	model=model,
	tools=[
		web_search,
		visit_webpage,
		final_answer,
	],
	max_steps=6,
	verbosity_level=2,
    grammar=None,
	planning_interval=None,
	name=None,
	description=None,
	prompt_templates=prompt_templates,
)

result = agent.run(
	"Indoor Service Robot"
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Indoor Service Robot                                                                                            │
│                                                                                                                 │
╰─ HfApiModel - meta-llama/Meta-Llama-3-8B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought:                                                                                                           
  Since I already know the description of indoor service robot, I will directly use the information I have.        
  DESCRIPTION: Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform    
tasks such as cleaning, delivery, security, and assistance.                                                        
  MODALITIES: rgb, depth                                                                                           
  CLASSES: furnitures, doors, obstacles, humans                                                                    
  METRIC: iou                                                                                                      
                                                                                                                   
Code:                                                                                                              
```py                                                                                                              
print(("Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform tasks such
as cleaning, delivery, security, and assistance.", ['rgb', 'depth'], ["furnitures", "doors", "obstacles",          
"humans"], ['iou']))                                                                                               
final_answer({                                                                                                     
  "DESCRIPTION": "Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform 
tasks such as cleaning, delivery, security, and assistance.",                                                      
  "MODALITIES": ['rgb', 'depth'],                                                                                  
  "CLASSES": ["furnitures", "doors", "obstacles", "humans"],                                                       
  "METRIC": ['iou'],                                                                                               
})                                                                                                                 
```<end_code>                                                                                                      

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  print(("Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform tasks   
  such as cleaning, delivery, security, and assistance.", ['rgb', 'depth'], ["furnitures", "doors", "obstacles",   
  "humans"], ['iou']))                                                                                             
  final_answer({                                                                                                   
    "DESCRIPTION": "Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to       
  perform tasks such as cleaning, delivery, security, and assistance.",                                            
    "MODALITIES": ['rgb', 'depth'],                                                                                
    "CLASSES": ["furnitures", "doors", "obstacles", "humans"],                                                     
    "METRIC": ['iou'],                                                                                             
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
('Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform tasks such as 
cleaning, delivery, security, and assistance.', ['rgb', 'depth'], ['furnitures', 'doors', 'obstacles', 'humans'], 
['iou'])

Out - Final answer: {'DESCRIPTION': 'Autonomous robots operating in indoor environments (homes, offices, hotels, 
campuses) to perform tasks such as cleaning, delivery, security, and assistance.', 'MODALITIES': ['rgb', 'depth'], 
'CLASSES': ['furnitures', 'doors', 'obstacles', 'humans'], 'METRIC': ['iou']}

[Step 1: Duration 3.44 seconds| Input tokens: 2,229 | Output tokens: 231]

In [15]:
description, modalities, obj_classes, metric = result['DESCRIPTION'], result['MODALITIES'], result['CLASSES'], result['METRIC']
result


{'DESCRIPTION': 'Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform tasks such as cleaning, delivery, security, and assistance.',
 'MODALITIES': ['rgb', 'depth'],
 'CLASSES': ['furnitures', 'doors', 'obstacles', 'humans'],
 'METRIC': ['iou']}

In [16]:
description

'Autonomous robots operating in indoor environments (homes, offices, hotels, campuses) to perform tasks such as cleaning, delivery, security, and assistance.'